# 03 — ResNet-50 Supervised Training (Authentic vs Tampered)

Fine-tunes pretrained `ResNet-50` (`ResNet50_Weights.DEFAULT`) for binary document layout classification (`Authentic` vs `Tampered`) using up to `20` epochs, `AdamW`, `CosineAnnealingLR`, and early stopping (`patience=8`) to maximize validation accuracy without overfitting.


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

EPOCHS = 20
EARLY_STOPPING_PATIENCE = 8
LEARNING_RATE = 1e-4
NUM_CLASSES = 2  # 0: Authentic, 1: Tampered
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f"ResNet-50 ready on {DEVICE} | max_epochs={EPOCHS}, patience={EARLY_STOPPING_PATIENCE}, lr={LEARNING_RATE}")


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

print("Training loop + CosineAnnealingLR defined. Connect train_loader and val_loader to run epochs.")
